# Step 1b · Feasibility check: Hurricane Florence (2018)

**Purpose**: before building the full dataset, use one hurricane to answer three questions.

1. **Outage ground truth**: are EAGLE-I's county-level outage records complete enough in the counties Florence hit?
2. **Flood**: can Sentinel-1 radar pick up the flood extent?
3. **Night lights**: do the drop and recovery in night lights match the real outages?

**Structure**

| Part | Content | Needs Earth Engine | Approx. time |
| --- | --- | --- | --- |
| A | EAGLE-I outage curves and county metrics (Figure 1) | No | 5–10 minutes (downloads about 1 GB) |
| B | Sentinel-1 flood map and flooded share by county | Yes | export task, 10–30 minutes |
| C | 3 counties + 1 control county: night lights against real outages (Figure 2) | Yes | about 3 minutes |
| D | First look at flood extent and outage duration (Figure 3) | No | 1 minute |
| E | Feasibility checklist | No | 1 minute |

**Requirements**

- An Earth Engine Cloud project, set as `PROJECT_ID` in cell 2 (part A does not use Earth Engine)
- Results are saved to the `us_flood_outage` folder; downloads and exports already done are skipped on a rerun

**Timing note**: Florence made landfall near Wrightsville Beach, North Carolina, at 7:15 AM EDT (11:15 UTC) on 14 September 2018. EAGLE-I timestamps are in UTC. VIIRS passes over at around 1:30–2:30 AM local time, so the 14 September night-light image is still pre-landfall; the first post-landfall image is 15 September.

In [ ]:
# 1. Environment, output folder, chart style
import os, time
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display, Image

try:                                    # in Colab: mount Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    OUT, TMP = "/content/drive/MyDrive/us_flood_outage", "/content"
except ImportError:                     # in local Jupyter: use the current folder
    OUT = "us_flood_outage" if os.path.isdir("us_flood_outage") else "../us_flood_outage"    # next to the notebook, or one level up
    TMP = f"{OUT}/tmp"
os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
print("Output folder:", OUT)

def tidy(df, n=1):
    # round numeric columns only, to keep tables readable
    return df.assign(**{c: df[c].round(n) for c in df.select_dtypes("number").columns})

# Chart style (shared by all figures): blue = power / outages, orange = night lights
C_POWER, C_NTL = "#2a78d6", "#eb6834"
INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

## Parameters

All event-specific settings are in this cell.

In [ ]:
# 2. Study parameters
PROJECT_ID = "ee-yourname"                       # Earth Engine Cloud project ID

EVENT    = "Florence"
LANDFALL = pd.Timestamp("2018-09-14 11:15")      # landfall time (UTC) = 7:15 AM EDT
STATES   = {"37": "NC", "45": "SC"}              # state FIPS code → state abbreviation

WIN_START, WIN_END = pd.Timestamp("2018-08-15"), pd.Timestamp("2018-10-15")   # outage data extraction window (UTC)
BASE_END           = pd.Timestamp("2018-09-10")                               # the pre-event baseline ends on this day
EVT_START, EVT_END = pd.Timestamp("2018-09-13"), pd.Timestamp("2018-10-05")   # window for computing outage metrics
OUT_THRESH = 0.10                                # "substantial outage": at least 10% of the county's customers are out
MIN_COVER  = 0.8                                 # night lights: a night is "clear" when at least 80% of the built-up pixels are usable
NTL_CYCLE  = 16 / 3                              # night lights: the satellite's viewing angle roughly repeats every 5.33 days (a third of its 16-day orbit cycle)

# Focus counties (3) and a control county (1, far from the track, to show how noisy night lights are on their own)
FOCUS   = {"37129": "New Hanover, NC (Wilmington, landfall)",
           "37049": "Craven, NC (New Bern, storm surge)",
           "37155": "Robeson, NC (Lumberton, river flooding)"}
CONTROL = {"45045": "Greenville, SC (control, about 400 km inland)"}

## A. Recorded outages (EAGLE-I)

EAGLE-I is a county-level count of "customers out", collected by Oak Ridge National Laboratory every 15 minutes from the public outage maps of electric utilities. Three properties of the record matter here:

- **Records with 0 customers out are not in the file.** So when a county has no record at some time, it can mean "nobody was out" or "the data was not collected". Below, a time at which all of NC + SC has no record at all is treated as a collection gap
- **Not every utility is covered.** Customer coverage in 2018 was about 82% in North Carolina and 96–99% in South Carolina. Outages at utilities that are not covered are not recorded, so the outage share is an underestimate
- **Customers are not people.** One "customer" is one meter (a household or a business)

In [ ]:
# 3. Download the EAGLE-I 2018 data; keep only NC + SC and the study window
EAGLEI_URL = "https://ndownloader.figshare.com/files/42547879"      # eaglei_outages_2018.csv (about 1 GB)

def download(url, path, label="", min_bytes=1000):
    # download a file to path; skip it if it already exists
    if os.path.exists(path) and os.path.getsize(path) >= min_bytes:
        return path
    print(f"Downloading {label or url} …")
    with requests.get(url, stream=True, timeout=180, headers={"User-Agent": "Mozilla/5.0"}) as r:
        r.raise_for_status()
        done, shown = 0, 0
        with open(path + ".part", "wb") as f:
            for block in r.iter_content(chunk_size=1 << 22):
                f.write(block); done += len(block)
                if done - shown >= 200e6:
                    shown = done; print(f"  {done / 1e6:,.0f} MB downloaded")
    if done < min_bytes:
        os.remove(path + ".part")
        raise RuntimeError(f"The downloaded file is too small ({done} bytes); the site may have blocked the request. Open {url} in a browser, download it manually, "
                           f"place it in the us_flood_outage folder under the same name, then rerun this cell.")
    os.replace(path + ".part", path)
    return path

SUB = f"{OUT}/eaglei_2018_NC_SC.csv"            # small filtered file, kept in the output folder and read directly on a rerun
if not os.path.exists(SUB):
    raw = f"{OUT}/eaglei_outages_2018.csv"      # a manually downloaded copy can be placed here
    if not os.path.exists(raw):
        raw = download(EAGLEI_URL, f"{TMP}/eaglei_outages_2018.csv", "EAGLE-I 2018 (about 1 GB, takes a few minutes)", 500e6)
    header = {c.strip().lower(): c for c in pd.read_csv(raw, nrows=0, encoding="utf-8-sig").columns}
    fips_col, time_col = header["fips_code"], header["run_start_time"]
    out_col = header.get("customers_out", header.get("sum"))
    parts = []
    for chunk in pd.read_csv(raw, usecols=[fips_col, out_col, time_col], dtype={fips_col: str},
                             encoding="utf-8-sig", chunksize=2_000_000):
        fips = chunk[fips_col].str.split(".").str[0].str.zfill(5)
        keep = fips.str[:2].isin(list(STATES))
        parts.append(pd.DataFrame({"fips": fips[keep], "customers_out": chunk.loc[keep, out_col],
                                   "time": chunk.loc[keep, time_col]}))
    ea = pd.concat(parts, ignore_index=True)
    for fmt in ("%Y-%m-%d %H:%M:%S", "%m/%d/%y %H:%M", None):      # the two timestamp styles that may appear in the file
        parsed = pd.to_datetime(ea["time"], format=fmt, errors="coerce")
        if parsed.notna().mean() > 0.99: break
    ea["time"] = parsed
    ea["customers_out"] = pd.to_numeric(ea["customers_out"], errors="coerce")
    n_bad = int(ea[["time", "customers_out"]].isna().any(axis=1).sum())
    ea = ea.dropna(subset=["time", "customers_out"])
    ea = ea[(ea["time"] >= WIN_START) & (ea["time"] < WIN_END)]
    ea.to_csv(SUB, index=False)
    print(f"Filtered: {len(ea):,} rows ({n_bad:,} unparseable rows dropped)")

ea = pd.read_csv(SUB, dtype={"fips": str}, parse_dates=["time"])
print(f"NC + SC, {ea['time'].min():%Y-%m-%d} to {ea['time'].max():%Y-%m-%d}: {len(ea):,} rows, {ea['fips'].nunique()} counties with records")
ea.head()

### How many customers each county has (the denominator)

Outage share = customers out ÷ total customers in the county. EAGLE-I ships with a county customer table (`MCC.csv`), but it cannot be used as is in North Carolina: for many counties, including the most populous ones, the table's customer count is far below the real figure, for example only 2,544 for New Hanover, 46,622 for Wake and 28,172 for Mecklenburg.

So this notebook uses a transparent estimate instead: **county customers = county population × statewide customers ÷ state population**. Population comes from the Census 2018 county estimates, and the statewide customer totals from EAGLE-I's `coverage_history.csv`. `MCC.csv` is still read in for comparison. How to set the denominator is a question to settle properly in Step 2; for now the aim is to get the curves to the right order of magnitude.

In [ ]:
# 4. Estimated customers per county (the denominator), compared with EAGLE-I's own MCC.csv
MCC_URL    = "https://ndownloader.figshare.com/files/42547708"
CENSUS_URL = "https://www2.census.gov/programs-surveys/popest/datasets/2010-2019/counties/totals/co-est2019-alldata.csv"
STATE_CUSTOMERS_2018 = {"NC": 5_179_052, "SC": 2_651_417}     # coverage_history.csv: statewide customer totals in 2018

pop = pd.read_csv(download(CENSUS_URL, f"{OUT}/co-est2019-alldata.csv", "Census county population estimates"),
                  encoding="latin-1", dtype={"STATE": str, "COUNTY": str})
pop = pop[pop["SUMLEV"] == 50].copy()
pop["fips"]  = pop["STATE"].str.zfill(2) + pop["COUNTY"].str.zfill(3)
pop["state"] = pop["STATE"].str.zfill(2).map(STATES)
cty = (pop.dropna(subset=["state"]).set_index("fips")[["CTYNAME", "state", "POPESTIMATE2018"]]
          .rename(columns={"CTYNAME": "name", "POPESTIMATE2018": "pop2018"}).sort_index())
cty["name"] = cty["name"].str.replace(" County", "", regex=False) + ", " + cty["state"]
per_person = pd.Series(STATE_CUSTOMERS_2018) / cty.groupby("state")["pop2018"].sum()
cty["customers_est"] = (cty["pop2018"] * cty["state"].map(per_person)).round()

mcc = pd.read_csv(download(MCC_URL, f"{OUT}/MCC.csv", "MCC.csv"), encoding="utf-8-sig")
mcc.columns = ["fips", "mcc_customers"]
mcc = mcc[pd.to_numeric(mcc["fips"], errors="coerce").notna()].copy()
mcc["fips"] = mcc["fips"].astype(str).str.split(".").str[0].str.zfill(5)
cty["mcc_customers"] = mcc.set_index("fips")["mcc_customers"].astype(float)
cty["mcc_ratio"] = cty["mcc_customers"] / cty["customers_est"]

print(f"{len(cty)} counties; customers per person: " + ", ".join(f"{s} {v:.2f}" for s, v in per_person.items()))
for s in STATES.values():
    sub = cty[cty["state"] == s]
    print(f"  {s}: MCC is below half of the estimate in {int((sub['mcc_ratio'] < 0.5).sum())} / {len(sub)} counties; "
          f"MCC state total {sub['mcc_customers'].sum():,.0f}, estimated total {sub['customers_est'].sum():,.0f}")
tidy(cty.loc[list(FOCUS) + list(CONTROL), ["name", "pop2018", "customers_est", "mcc_customers", "mcc_ratio"]], 2)

In [ ]:
# 5. 15-minute panel → outage share → outage metrics per county
STEP_H = 0.25                                                   # one time slot = 15 minutes
grid = pd.date_range(WIN_START, WIN_END, freq="15min", inclusive="left")
ea["slot"] = ea["time"].dt.floor("15min")
wide = (ea.groupby(["slot", "fips"])["customers_out"].max().unstack("fips")
          .reindex(index=grid, columns=cty.index))

# No record = 0 customers out; but if all of NC + SC has no record at some time, treat it as a collection gap
has_rows = pd.Series(grid.isin(ea["slot"]), index=grid)
wide = wide.fillna(0.0)
wide.loc[~has_rows.values, :] = np.nan
wide = wide.ffill(limit=8)                                      # fill gaps of up to 2 hours with the previous value

g = (~has_rows).loc[EVT_START:EVT_END]
longest_gap_h = float(g.groupby((~g).cumsum()).sum().max() * STEP_H)
print(f"Collection gaps in the event window: {g.sum() * STEP_H:.1f} hours in total, longest {longest_gap_h:.1f} hours")

raw_frac = wide.div(cty["customers_est"], axis=1)
frac = raw_frac.clip(upper=1.0)                                 # outage share (0–1)
print(f"Counties where customers out exceed the estimated customer total: {int((raw_frac.max() > 1).sum())} (capped at 100%)")

ev = frac.loc[EVT_START:EVT_END]
above = ev >= OUT_THRESH
metrics = pd.DataFrame({
    "name": cty["name"], "state": cty["state"],
    "peak_customers_out": wide.loc[ev.index].max(),
    "peak_pct": ev.max() * 100,                                 # peak outage share (%)
    "peak_time": ev.fillna(0).idxmax(),
    "hours_above": above.sum() * STEP_H,                        # cumulative hours with an outage share ≥ 10%
    "customer_hours": wide.loc[ev.index].sum() * STEP_H,        # customer-hours without power
})
# Restoration time: the first time after the peak that stays below 10% for 24 hours (small later outages do not stretch it)
calm = (ev < OUT_THRESH).astype(float)[::-1].rolling(96, min_periods=96).min()[::-1].eq(1)
after_peak = pd.DataFrame(ev.index.values[:, None] >= metrics["peak_time"].values[None, :], index=ev.index, columns=ev.columns)
restored = calm & after_peak
metrics["days_to_restore"] = ((restored.idxmax() - LANDFALL) / pd.Timedelta(days=1)).where(restored.any() & above.any())
metrics.to_csv(f"{OUT}/{EVENT.lower()}_outage_metrics_by_county.csv")

print(f"Counties with a peak outage ≥ 10%: {int((metrics['peak_pct'] >= 10).sum())}; "
      f"≥ 10% for more than 24 hours: {int((metrics['hours_above'] >= 24).sum())}; "
      f"for more than 7 days: {int((metrics['hours_above'] >= 168).sum())}")
tidy(metrics.sort_values("hours_above", ascending=False).head(15))

In [ ]:
# 6. Figure 1: statewide outage curves + the 15 counties with the longest outages
x_from = LANDFALL.normalize() - pd.Timedelta(days=3)
state_out = wide.loc[x_from:EVT_END].T.groupby(cty["state"]).sum(min_count=1).T / 1000      # thousands of customers
top = metrics.sort_values("hours_above", ascending=False).head(15)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.8), gridspec_kw={"width_ratios": [1.2, 1], "wspace": 0.42})

ax1.axvline(LANDFALL, color=FAINT, lw=1)
for s, c in zip(["NC", "SC"], [C_POWER, C_NTL]):
    ax1.plot(state_out.index, state_out[s], color=c, lw=2,
             label={"NC": "North Carolina", "SC": "South Carolina"}[s])
    t_pk = state_out[s].idxmax()
    ax1.annotate(f"{state_out[s].max():,.0f}k", (t_pk, state_out[s].max()), xytext=(6, 2),
                 textcoords="offset points", color=INK, fontsize=9)
ax1.set_ylim(bottom=0); ax1.set_xlim(x_from, EVT_END)
ax1.annotate("Landfall", (LANDFALL, ax1.get_ylim()[1]), xytext=(-4, -11), textcoords="offset points",
             color=MUTED, fontsize=9, ha="right")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=4))
ax1.set_title(f"Customers without power, Hurricane {EVENT} (thousands)")
ax1.set_xlabel("2018 (UTC)"); ax1.legend(loc="upper right")

y = np.arange(len(top))
ax2.barh(y, top["hours_above"], height=0.55, color=C_POWER)
for yi, h in zip(y, top["hours_above"]):
    ax2.annotate(f"{h:,.0f} h", (h, yi), xytext=(4, 0), textcoords="offset points", va="center",
                 color=INK, fontsize=9)
ax2.set_yticks(y); ax2.set_yticklabels(top["name"], color=INK); ax2.invert_yaxis()
ax2.grid(axis="y", visible=False); ax2.set_xlim(0, top["hours_above"].max() * 1.15)
ax2.set_title(f"Hours with {OUT_THRESH:.0%}+ of customers out, top 15 counties")
ax2.set_xlabel("Hours")
fig.savefig(f"{OUT}/fig1_{EVENT.lower()}_outages.png"); plt.show()

print("Statewide peaks (thousands of customers):", {s: round(float(state_out[s].max())) for s in ["NC", "SC"]})
print("For comparison: the U.S. DOE situation report of the afternoon of September 15 gives 24-hour peaks of about 846 thousand in NC and 167 thousand in SC.")

**Notes on Figure 1**

- The statewide peaks on the left can be compared with the DOE situation reports, whose figures also come from EAGLE-I
- The right panel shows one of the outcome variables used later in the modeling

## B. Flood extent (Sentinel-1 radar)

**Method**: calm water acts like a mirror and reflects the radar signal away, so flooded areas turn dark in radar images. The approach compares radar backscatter (VV polarization) before and after the event; a pixel counts as flooded when both conditions hold:

1. It is more than 3 dB darker after than before
2. Its post-event value is below −15 dB, a level typical of open water

Permanent water (water present ≥ 10 months a year), slopes above 5°, and patches smaller than 8 pixels are then removed. Before and after are compared only within the **same orbit**, to avoid false changes caused by different viewing angles.

**Known limitations**

- **Floods in built-up areas are missed**: strong reflections from buildings swamp the water signal. Outages and night lights are concentrated in exactly those areas, so the flooded share is reported separately for "whole county" and "built-up land" below
- **Flooded forest is missed**: double-bounce reflection between trunks and the water surface makes the backscatter stronger, not weaker. The North Carolina coastal plain has large forested wetlands
- **The September harvest can cause false alarms**: harvested fields also turn darker, and rule 2 only filters out part of them

In [ ]:
# 7. [EE] Authenticate and initialize Earth Engine
import ee
ee.Authenticate()                         # follow the link to authorize
ee.Initialize(project=PROJECT_ID)
print("Earth Engine initialized")

In [ ]:
# 8. [EE] County boundaries + Sentinel-1 image inventory
AOI = ee.Geometry.Rectangle([-84.4, 32.0, -75.4, 36.6])            # covers NC + SC
counties = (ee.FeatureCollection("TIGER/2018/Counties")
              .filter(ee.Filter.inList("STATEFP", list(STATES))))

S1_BEFORE = ("2018-08-13", "2018-09-10")     # before: about 4 weeks, take the median
S1_AFTER  = ("2018-09-14", "2018-09-28")     # after: two weeks from landfall, take the darkest value (rivers crested one to two weeks after landfall)

s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(AOI)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .select("VV")
        .map(lambda img: img.updateMask(img.gt(-30))))             # remove noise at the image edges
before, after = s1.filterDate(*S1_BEFORE), s1.filterDate(*S1_AFTER)

def inventory(col, label):
    rows = col.reduceColumns(ee.Reducer.toList(3),
                             ["system:time_start", "orbitProperties_pass", "relativeOrbitNumber_start"]
                            ).get("list").getInfo()
    df = pd.DataFrame(rows, columns=["t", "pass", "rel_orbit"])
    df["date"] = pd.to_datetime(df["t"], unit="ms").dt.normalize()
    df["rel_orbit"] = df["rel_orbit"].astype(int)
    df["period"] = label
    return df

inv = pd.concat([inventory(before, "before"), inventory(after, "after")], ignore_index=True)
inv.to_csv(f"{OUT}/{EVENT.lower()}_s1_inventory.csv", index=False)

In [ ]:
# 9. Which orbits have images both before and after (only these can be used for change detection)
tab = inv.groupby(["rel_orbit", "pass", "period"]).size().unstack("period", fill_value=0)
for c in ["before", "after"]:
    if c not in tab: tab[c] = 0
tab["after_dates"] = (inv[inv["period"] == "after"].groupby(["rel_orbit", "pass"])["date"]
                        .apply(lambda d: ", ".join(sorted({f"{x:%m-%d}" for x in d}))))
orbits = sorted({int(r) for r in tab[(tab["before"] > 0) & (tab["after"] > 0)].index.get_level_values("rel_orbit")})
print("Usable orbits:", orbits)
tab[["before", "after", "after_dates"]]

In [ ]:
# 10. [EE] Flood map: change detection orbit by orbit, then merge
DIFF_DB, WATER_DB = -3.0, -15.0            # thresholds for the two rules (dB)
SMOOTH_M, SLOPE_MAX, MIN_PIXELS = 50, 5, 8 # smoothing radius (m), maximum slope (degrees), minimum patch size (pixels)
GHSL_YEAR, BUILT_MIN_M2 = 2015, 1000       # built-up: at least 1,000 m² of built surface in a 100 m cell (10%)

def flood_one_orbit(r):
    same = ee.Filter.eq("relativeOrbitNumber_start", r)
    b = before.filter(same).median().focalMedian(SMOOTH_M, "circle", "meters")
    a = after.filter(same).min().focalMedian(SMOOTH_M, "circle", "meters")
    return a.subtract(b).lt(DIFF_DB).And(a.lt(WATER_DB)).rename("flood")

flood_raw  = ee.ImageCollection([flood_one_orbit(r) for r in orbits]).max()      # flooded if any orbit flags it
perm_water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("seasonality").gte(10).unmask(0)
slope      = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003"))
candidate  = flood_raw.And(perm_water.Not()).And(slope.lt(SLOPE_MAX))
big_enough = candidate.selfMask().connectedPixelCount(MIN_PIXELS + 2, True).gte(MIN_PIXELS).unmask(0)
flood      = candidate.And(big_enough).rename("flood")              # 1 = flooded, 0 = not flooded; masked where the radar has no coverage

ghsl    = (ee.ImageCollection("JRC/GHSL/P2023A/GHS_BUILT_S")
             .filterDate(f"{GHSL_YEAR}-01-01", f"{GHSL_YEAR + 1}-01-01").first().select("built_surface"))
builtup = ghsl.gte(BUILT_MIN_M2).unmask(0)

# Four areas per county (km²): flooded, land covered by the radar, built-up land within it, flooded built-up land
km2  = ee.Image.pixelArea().divide(1e6)
land = flood.gte(0).And(perm_water.Not())
stats_img = ee.Image.cat([
    flood.multiply(km2).rename("flood_km2"),
    land.multiply(km2).rename("land_km2"),
    land.And(builtup).multiply(km2).rename("built_km2"),
    flood.And(builtup).multiply(km2).rename("built_flood_km2"),
])
county_stats = stats_img.reduceRegions(collection=counties.select(["GEOID", "NAME", "STATEFP", "ALAND"]),
                                       reducer=ee.Reducer.sum(), scale=30, tileScale=4)

# the export task runs on Google's servers and continues if the Colab session ends
FLOOD_NAME = f"{EVENT.lower()}_flood_by_county"
FLOOD_CSV  = f"{OUT}/{FLOOD_NAME}.csv"
active = [t for t in ee.data.getTaskList()
          if t.get("description") == FLOOD_NAME and t.get("state") in ("READY", "RUNNING")]
if os.path.exists(FLOOD_CSV):
    print("Result already in Drive, skipping the export:", FLOOD_CSV)
elif active:
    print("Export task already running; not submitting again")
else:
    ee.batch.Export.table.toDrive(
        collection=county_stats, description=FLOOD_NAME, folder="us_flood_outage", fileNamePrefix=FLOOD_NAME,
        fileFormat="CSV", selectors=["GEOID", "NAME", "STATEFP", "ALAND", "flood_km2", "land_km2",
                                     "built_km2", "built_flood_km2"]).start()
    print("Export task submitted. Progress: https://code.earthengine.google.com/tasks")

### Visual check of the flood map

The next two cells display the flood map, which is best judged at neighborhood zoom: Earth Engine computes at screen resolution, so when zoomed far out the smoothing and the small-patch filter behave differently, and the flood extent does not look like the 30 m result used for the statistics.

The visual check covers three points: whether the flagged water follows river channels and low ground rather than rectangular field boundaries; whether false flood appears in stripes along image edges; and whether anything is flagged inside towns such as downtown New Bern or Lumberton, since nothing flagged there at all would mean that floods in built-up areas are badly missed.

In [ ]:
# 11. [EE] Static quicklook: around Burgaw, Pender County (Northeast Cape Fear River, one of the areas Florence flooded worst)
THUMB_REGION = ee.Geometry.Rectangle([-78.15, 34.35, -77.75, 34.70])

def fetch_thumb(img, name):
    # request a PNG from Earth Engine; on failure, print Earth Engine's own error message
    try:
        url = img.getThumbURL({"region": THUMB_REGION, "dimensions": 1100, "format": "png"})
        resp = requests.get(url, timeout=300)
    except Exception as e:
        print(f"[{name}] ✗ {e}"); return None
    if resp.status_code != 200:
        try:
            msg = resp.json()["error"]["message"]
        except Exception:
            msg = resp.text[:500]
        print(f"[{name}] ✗ HTTP {resp.status_code}: {msg}"); return None
    return resp.content

layers = {
    # all three layers need a palette: that makes each one a 3-band RGB image, so they can be stacked
    "S1 after": after.min().visualize(min=-25, max=0, palette=["000000", "ffffff"]),   # base layer: post-event radar image (darker = more water-like)
    "permanent water": perm_water.selfMask().visualize(palette=["0d366b"]),            # dark blue: permanent water
    "flood": flood.selfMask().visualize(palette=["3987e5"]),                           # bright blue: flood
}
img_bytes = fetch_thumb(ee.ImageCollection(list(layers.values())).mosaic(), "all layers")
if img_bytes is None:                        # try layer by layer to see which one fails
    good = [name for name, img in layers.items() if fetch_thumb(img, name) is not None]
    print("Layers that work:", good)
    if good:
        img_bytes = fetch_thumb(ee.ImageCollection([layers[n] for n in good]).mosaic(), "working layers")
if img_bytes is not None:
    png = f"{OUT}/fig_{EVENT.lower()}_flood_quicklook.png"
    with open(png, "wb") as f:
        f.write(img_bytes)
    display(Image(filename=png, width=700))

In [ ]:
# 12. [EE] Interactive map (toggle layers, zoom in)
import geemap
m = geemap.Map(center=[34.55, -77.95], zoom=11)
m.addLayer(before.median(), {"min": -25, "max": 0}, "S1 before (VV median)", False)
m.addLayer(after.min(), {"min": -25, "max": 0}, "S1 after (VV minimum)")
m.addLayer(perm_water.selfMask(), {"palette": ["0d366b"]}, "Permanent water")
m.addLayer(flood.selfMask(), {"palette": ["3987e5"]}, "Flood (Sentinel-1)")
try:      # reference layer: Global Flood Database (Tellman et al., MODIS 250 m); skipped if not found
    gfd = (ee.ImageCollection("GLOBAL_FLOOD_DB/MODIS_EVENTS/V1")
             .filterBounds(AOI).filterDate("2018-09-01", "2018-10-15"))
    n_gfd = gfd.size().getInfo()
    if n_gfd:
        m.addLayer(gfd.select("flooded").max().selfMask(), {"palette": ["e87ba4"]},
                   "Reference: Global Flood Database (MODIS)", False)
    print(f"Global Flood Database events in this period: {n_gfd}")
except Exception as e:
    print("Reference layer not loaded:", e)
m.addLayer(counties.style(color="ffffff", fillColor="00000000", width=1), {}, "Counties")
m

## C. Night lights against real outages (3 focus counties + 1 control county)

Night lights are cleaned more strictly than in the earlier project on Philippine typhoons (`ph-typhoon-outage-transfer`): keep only high-quality, confidently clear pixels with no cloud shadow or thin cirrus; look only at built-up areas; a night counts as "clear" only when at least 80% of the built-up pixels are usable.

Unlike in the Philippines, recorded outages are available to compare against. Both curves are converted to "percent of normal" and drawn on the same vertical axis:

- **Share with power** = 100% − outage share, averaged around the satellite overpass (6–8 UTC, i.e. 2–4 AM local time)
- **Night lights** = that night's radiance ÷ the normal radiance at the same viewing angle

**Why adjust for viewing angle**: the satellite looks at the same city from a different angle each night, and the angle roughly repeats every 5.33 days. The same lights can look about 20% brighter or dimmer from one angle to another, which has nothing to do with outages. So the notebook first fits "how normal radiance cycles with viewing angle" on the clear nights before landfall, then compares each night with the normal radiance for its angle. This roughly halves the normal night-to-night variation (see `noise_raw_pct` and `noise_adj_pct` in the table below).

In [ ]:
# 13. [EE] Extract the mean built-up radiance for each county and night
NTL_START, NTL_END = "2018-08-01", "2018-11-01"
NTL_COL, NTL_BAND = "NASA/VIIRS/002/VNP46A2", "DNB_BRDF_Corrected_NTL"
QF_BAND, CLOUD_BAND = "Mandatory_Quality_Flag", "QF_Cloud_Mask"
BUILT_MIN_FRAC = 0.10                      # a 500 m pixel counts as built-up when at least 10% of it is built

raw_ntl = ee.ImageCollection(NTL_COL)
proj = raw_ntl.first().select(NTL_BAND).projection()

def clean(img):
    ok = img.select(QF_BAND).lte(1)                                   # 0/1 = high quality
    qf = img.select(CLOUD_BAND)
    clear  = qf.rightShift(6).bitwiseAnd(3).eq(0)                     # bits 6–7: keep "confident clear" only
    shadow = qf.rightShift(8).bitwiseAnd(1).eq(0)                     # bit 8: no cloud shadow
    cirrus = qf.rightShift(9).bitwiseAnd(1).eq(0)                     # bit 9: no thin cirrus
    return (img.select(NTL_BAND).updateMask(ok.And(clear).And(shadow).And(cirrus)).rename("ntl")
               .copyProperties(img, ["system:time_start"]))

ntl = raw_ntl.filterDate(NTL_START, NTL_END).map(clean)

# GHSL built-up surface (m² per 100 m cell) → built-up fraction of each 500 m pixel
built500 = (ghsl.divide(10000).reduceResolution(ee.Reducer.mean(), maxPixels=64).reproject(proj))
settlement = built500.gte(BUILT_MIN_FRAC)

show = {**FOCUS, **CONTROL}
show_fc = counties.filter(ee.Filter.inList("GEOID", list(show))).select(["GEOID"])
elig_fc = settlement.selfMask().reduceRegions(collection=show_fc, reducer=ee.Reducer.count(), crs=proj)
n_elig = {f["properties"]["GEOID"]: f["properties"]["count"] for f in elig_fc.getInfo()["features"]}
print("Built-up pixels per county:", n_elig)

mean_count = ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True)
def nightly(img):
    img = ee.Image(img)
    day = img.date().format("YYYY-MM-dd")
    stats = img.updateMask(settlement).reduceRegions(collection=show_fc, reducer=mean_count, crs=proj)
    return stats.map(lambda f: f.set("date", day))

rows = ee.FeatureCollection(ntl.map(nightly)).flatten().select(["GEOID", "date", "mean", "count"], None, False)
ntl_df = pd.DataFrame([f["properties"] for f in rows.getInfo()["features"]])
ntl_df = ntl_df.rename(columns={"ntl_mean": "mean", "ntl_count": "count"}).reindex(columns=["GEOID", "date", "mean", "count"])
ntl_df["n_elig"] = ntl_df["GEOID"].map(n_elig)
ntl_df.to_csv(f"{OUT}/{EVENT.lower()}_ntl_focus_counties.csv", index=False)
print(f"Extraction done: {len(ntl_df)} rows")

In [ ]:
# 14. Figure 2: night lights (adjusted for viewing angle) against the true share of customers with power
ntl_df = pd.read_csv(f"{OUT}/{EVENT.lower()}_ntl_focus_counties.csv", dtype={"GEOID": str}, parse_dates=["date"])
ntl_df["clear"] = (ntl_df["count"].fillna(0) / ntl_df["n_elig"]) >= MIN_COVER
show = {**FOCUS, **CONTROL}
days = pd.date_range(ntl_df["date"].min(), ntl_df["date"].max(), freq="D")
first_night = LANDFALL.normalize() + pd.Timedelta(days=1)          # first night after landfall

def angle_terms(dates):
    # the viewing-angle cycle: a constant plus one cosine–sine pair
    phase = 2 * np.pi * (pd.DatetimeIndex(dates) - days[0]).days.values / NTL_CYCLE
    return np.column_stack([np.ones(len(phase)), np.cos(phase), np.sin(phase)])

def expected_ntl(normal, dates):
    # fit how radiance varies with viewing angle on normal nights; return the radiance expected on `dates` under normal conditions
    if len(normal) < 10:                                           # too few clear nights: fall back to the median
        return np.full(len(dates), normal.median())
    coef = np.linalg.lstsq(angle_terms(normal.index), np.log(normal.values), rcond=None)[0]
    return np.exp(angle_terms(dates) @ coef)

panel, summary = {}, []
for fips, label in show.items():
    d = ntl_df[(ntl_df["GEOID"] == fips) & ntl_df["clear"]].set_index("date")["mean"].sort_index()
    base = d.loc[:BASE_END]                                        # clear nights before landfall
    ntl_pct = d / expected_ntl(base, d.index) * 100
    # noise: leave each pre-storm clear night out in turn, predict it from the others, and see how far off it is
    left_out = [base.iloc[i] / expected_ntl(base.drop(base.index[i]), base.index[[i]])[0] for i in range(len(base))]
    s = frac[fips]
    s = s[(s.index.hour >= 6) & (s.index.hour < 8)]                # around the satellite overpass (UTC)
    power_pct = (1 - s.groupby(s.index.normalize()).mean()) * 100
    df = pd.DataFrame({"power_pct": power_pct, "ntl_pct": ntl_pct}).reindex(days)
    panel[fips] = df

    post  = df.loc[first_night:]
    both  = df.loc["2018-09-01":"2018-10-14"].dropna()
    first = post["ntl_pct"].first_valid_index()
    summary.append({
        "county": label.split(" (")[0], "role": "control" if fips in CONTROL else "focus",
        "baseline_clear_nights": int(base.notna().sum()),
        "noise_raw_pct": round(float((base / base.median() * 100).std()), 1),   # normal variation before the viewing-angle adjustment (standard deviation)
        "noise_adj_pct": round(float(np.std(left_out, ddof=1) * 100), 1),       # normal variation after the adjustment
        "clear_nights_first_7": int(post["ntl_pct"].iloc[:7].notna().sum()),    # clear nights among the first 7 after landfall
        "first_clear_night": first.strftime("%m-%d") if first is not None else "none",
        "ntl_pct_then": round(float(df.loc[first, "ntl_pct"]), 0) if first is not None else np.nan,
        "power_pct_then": round(float(df.loc[first, "power_pct"]), 0) if first is not None else np.nan,
        "min_power_pct": round(float(post["power_pct"].min()), 0),
        "pearson_r": (round(float(both["power_pct"].corr(both["ntl_pct"])), 2)
                      if len(both) >= 8 and both["power_pct"].std() > 0 else np.nan),
        "n_pairs": len(both),
    })
ntl_summary = pd.DataFrame(summary).set_index("county")

x0, x1 = pd.Timestamp("2018-09-01"), pd.Timestamp("2018-10-14")
ymax = float(np.clip(max(p.loc[x0:x1, "ntl_pct"].max() for p in panel.values()) * 1.08, 130, 200))
fig, axes = plt.subplots(len(show), 1, figsize=(10, 2.25 * len(show)), sharex=True, sharey=True)
for ax, (fips, label) in zip(axes, show.items()):
    df = panel[fips].loc[x0:x1]
    n = df["ntl_pct"].dropna().clip(upper=ymax)
    cloudy = df.index[df["ntl_pct"].isna()]
    ax.axvline(LANDFALL, color=FAINT, lw=1)
    ax.plot(cloudy, np.full(len(cloudy), ymax * 0.045), ls="none", marker="|", ms=7, mew=1.2, color=FAINT,
            label="No usable night-light image (cloud)")
    ax.plot(df.index, df["power_pct"], color=C_POWER, lw=2, label="Customers with power, 2–4 AM local (EAGLE-I)")
    ax.plot(n.index, n.values, color=C_NTL, lw=1, alpha=0.4)
    ax.plot(n.index, n.values, ls="none", marker="o", ms=7, mfc=C_NTL, mec=SURFACE, mew=1.5,
            label="Night lights on clear nights, adjusted for viewing angle (Black Marble)")
    ax.set_title(label); ax.set_ylim(0, ymax); ax.set_yticks([0, 50, 100])
    ax.grid(axis="x", visible=False); ax.tick_params(axis="x", length=0 if ax is not axes[-1] else 3.5)
axes[0].annotate("Landfall", (LANDFALL, ymax), xytext=(-4, -11), textcoords="offset points",
                 color=MUTED, fontsize=9, ha="right")
axes[len(show) // 2].set_ylabel("Percent of pre-storm normal")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axes[-1].xaxis.set_major_locator(mdates.DayLocator(interval=7))
axes[-1].set_xlabel("2018")
handles, labels = axes[0].get_legend_handles_labels()
order = [1, 2, 0]
fig.legend([handles[i] for i in order], [labels[i] for i in order], loc="lower left",
           bbox_to_anchor=(0.06, 1.0), ncol=2, columnspacing=1.6, handletextpad=0.5)
fig.tight_layout()
fig.savefig(f"{OUT}/fig2_{EVENT.lower()}_ntl_vs_outage.png"); plt.show()
ntl_summary

**Notes on Figure 2 and the table above**

- Whether the orange dots fall and recover with the blue line is the direct test of whether night lights can stand in for recorded outages
- `clear_nights_first_7`: clear nights among the first 7 after landfall. Hurricanes are often followed by days of cloud; with 0 or 1, night lights miss the worst days of the outage
- `noise_raw_pct` and `noise_adj_pct`: night-to-night variation on normal nights, before and after the viewing-angle adjustment. A drop needs to be clearly larger than `noise_adj_pct` (roughly twice as large) to be detectable
- The control county's blue line stays close to 100%, and the scatter of its orange dots gives the noise level. A deep drop in the control county right after landfall points to cloud that was not screened out
- `pearson_r` is unstable when clear nights are few; the figure is the better guide

## D. First look at flood extent and outage duration

This cell needs the export task of part B to have finished (progress at <https://code.earthengine.google.com/tasks>). It only looks at the direction: do counties with more flooding have longer outages? It does **not** show causation, because heavily flooded counties tend to have strong winds as well; Step 4 separates the two.

In [ ]:
# 15. Read the county flood statistics, merge them with the outage metrics, draw Figure 3
FLOOD_CSV = f"{OUT}/{EVENT.lower()}_flood_by_county.csv"
for _ in range(20):                          # Drive sync can take a minute or two
    if os.path.exists(FLOOD_CSV): break
    time.sleep(15)
if not os.path.exists(FLOOD_CSV):
    raise FileNotFoundError("Flood statistics file not found yet. Check that the export task has finished, then rerun this cell.")

flood_cty = pd.read_csv(FLOOD_CSV, dtype={"GEOID": str})
flood_cty["GEOID"] = flood_cty["GEOID"].str.zfill(5)
flood_cty = flood_cty.set_index("GEOID")
flood_cty["s1_cover_pct"]    = (100 * flood_cty["land_km2"] / (flood_cty["ALAND"] / 1e6)).clip(upper=100)
flood_cty["flood_pct"]       = 100 * flood_cty["flood_km2"] / flood_cty["land_km2"]              # flooded share of all county land
flood_cty["built_flood_pct"] = 100 * flood_cty["built_flood_km2"] / flood_cty["built_km2"]       # flooded share of built-up land

m_all = metrics.join(flood_cty[["s1_cover_pct", "flood_km2", "flood_pct", "built_flood_pct"]])
hit = m_all[(m_all["peak_pct"] >= 10) & m_all["flood_pct"].notna()]
rho = hit["flood_pct"].corr(hit["hours_above"], method="spearman")

fig, ax = plt.subplots(figsize=(6.6, 4.8))
ax.scatter(hit["flood_pct"], hit["hours_above"], s=46, color=C_POWER, edgecolor=SURFACE, linewidth=1.3)
xmax = float(hit["flood_pct"].max())
for fips, r in hit.nlargest(4, "flood_pct").iterrows():
    right = r["flood_pct"] > 0.8 * xmax
    ax.annotate(r["name"], (r["flood_pct"], r["hours_above"]), xytext=(-7 if right else 7, 5),
                ha="right" if right else "left", textcoords="offset points", color=INK, fontsize=9)
ax.set_xlim(-0.03 * xmax, 1.08 * xmax); ax.set_ylim(bottom=0)
ax.set_xlabel("Share of county land flagged as flooded by Sentinel-1 (%)")
ax.set_ylabel(f"Hours with at least {OUT_THRESH:.0%} of customers out")
ax.set_title(f"Flood extent and outage duration by county, Hurricane {EVENT}", pad=22)
ax.text(0, 1.03, f"Counties with a peak outage of at least 10% (n = {len(hit)})", transform=ax.transAxes,
        color=MUTED, fontsize=9)
fig.savefig(f"{OUT}/fig3_{EVENT.lower()}_flood_vs_outage.png"); plt.show()

print(f"Spearman correlation: {rho:.2f} ({len(hit)} counties)")
print(f"Median of built-up flooded share ÷ county flooded share: {(hit['built_flood_pct'] / hit['flood_pct']).median():.2f} (well below 1 means floods in built-up areas are being missed)")
tidy(m_all.sort_values("flood_pct", ascending=False).head(10)[
    ["name", "s1_cover_pct", "flood_km2", "flood_pct", "built_flood_pct", "peak_pct", "hours_above"]])

**Reference for the flood ranking**: according to public reports, Florence's worst flooding was in southeastern North Carolina and northeastern South Carolina, for example Pender, Duplin, Jones, Craven, Robeson, Columbus, Bladen and Brunswick, plus Horry, Marion and Dillon in South Carolina. Inland farming counties near the top of the table would point to false alarms from the harvest, which the −15 dB rule (`WATER_DB`) is meant to filter out.

## E. Feasibility checklist

The thresholds were set before looking at the results, to decide whether to keep Florence as the anchor event.

In [ ]:
# 16. Feasibility checklist
def verdict(passed):
    return "pending" if passed is None else ("pass" if passed else "fail")

n_long = int((metrics["hours_above"] >= 24).sum())
checks = [("Outages", "Counties with ≥ 10% of customers out for more than 24 hours", n_long, "≥ 30", n_long >= 30),
          ("Outages", "Longest data collection gap in the event window (hours)", round(longest_gap_h, 1), "≤ 6", longest_gap_h <= 6)]

inv_path = f"{OUT}/{EVENT.lower()}_s1_inventory.csv"
if "inv" not in globals() and os.path.exists(inv_path):
    inv = pd.read_csv(inv_path, parse_dates=["date"])
if "inv" in globals():
    within5 = inv[(inv["period"] == "after") & (inv["date"] <= LANDFALL.normalize() + pd.Timedelta(days=5))]
    n_days = int(within5["date"].nunique())
    checks.append(("Flood", "Days with Sentinel-1 images within 5 days of landfall", n_days, "≥ 1", n_days >= 1))
else:
    checks.append(("Flood", "Days with Sentinel-1 images within 5 days of landfall", "—", "≥ 1", None))

if "flood_cty" in globals():
    cover = float(flood_cty.loc[list(FOCUS), "s1_cover_pct"].min())
    n_fl  = int((flood_cty["flood_pct"] >= 1).sum())
    checks += [("Flood", "Share of focus-county land covered by Sentinel-1 (minimum, %)", round(cover), "≥ 90", cover >= 90),
               ("Flood", "Counties with a flooded share ≥ 1%", n_fl, "≥ 10", n_fl >= 10)]
else:
    checks += [("Flood", "Share of focus-county land covered by Sentinel-1 (minimum, %)", "—", "≥ 90", None),
               ("Flood", "Counties with a flooded share ≥ 1%", "—", "≥ 10", None)]

if "ntl_summary" in globals():
    foc = ntl_summary[ntl_summary["role"] == "focus"]
    k_clear = int((foc["clear_nights_first_7"] >= 2).sum())
    k_corr  = int((foc["pearson_r"] >= 0.5).sum())
    checks += [("Night lights", "Focus counties with at least 2 clear nights in the first 7 after landfall", f"{k_clear} / {len(foc)}", "≥ 2", k_clear >= 2),
               ("Night lights", "Focus counties where night lights and the true share with power correlate at ≥ 0.5", f"{k_corr} / {len(foc)}", "≥ 2", k_corr >= 2)]
else:
    checks += [("Night lights", "Focus counties with at least 2 clear nights in the first 7 after landfall", "—", "≥ 2", None),
               ("Night lights", "Focus counties where night lights and the true share with power correlate at ≥ 0.5", "—", "≥ 2", None)]

check_tab = pd.DataFrame(checks, columns=["Part", "Indicator", "Value", "Threshold", "passed"])
check_tab["Verdict"] = check_tab.pop("passed").map(verdict)
check_tab.to_csv(f"{OUT}/{EVENT.lower()}_feasibility_checklist.csv", index=False)
check_tab

## What came next

The checklist in part E decides whether the three data sources are good enough to go on. Step 2c then extends the outage labels from Florence to nine hurricanes.